# Introducing DYCOVE

Short for DYnamic COastal VEgetation, the DYCOVE model takes as input a set of vegetation characteristics for any number of species of interest and computes colonization, growth, and mortality of those species based on hydrodynamic (and morphodynamic) conditions present in each grid cell in the numerical model. In turn, the evolving vegetation changes the flow resistance in the numerical model.

This notebook presents a simple ANUGA hydrodynamic model, which we couple with DYCOVE to model vegetation dynamics over time.

The GitHub repository for DYCOVE is located at https://github.com/Ecomomo-lab/dycove-model.

The DYCOVE documentation page is here: https://ecomomo-lab.github.io/dycove-model/index.html.

This example corresponds to the Python file `tide_channel_ANUGA.py` located in the `examples/ANUGA/tide_channel/` directory of the repository, along with supporting input files.

This example consists of a simple, symmetrical beach slope with a channel that bisects the dune and connects to a lagoon.
A simple mesh is developed using one of ANUGA's built-in functions that creates a rectangular mesh of structured triangles.
A tidal boundary condition is imposed on the left (ocean) side, that propagates through the channel and fills the lagoon behind the dune.
A generic vegetation species, loosely based on the attributes of Spartina anglica and Salicornia spp., is added to the model via a set of parameters defined in the file `veg1.json`.
After the simulation is finished, we can inspect a variety of 2-D model results using the `ModelPlotter` class.

**To run this notebook, select "Connect to a hosted runtime" via the dropdown menu in the upper-right.**

# Step 1: Install DYCOVE

Clone DYCOVE from GitHub to get access to the source code, examples, and more. Install with pip.


In [ ]:
import os
import sys

In [ ]:
!git clone https://github.com/Ecomomo-lab/dycove-model.git
%cd dycove-model
!pip install .

# # Extract local package after uploading, move into root folder, install
# !tar -xf dycove-model.tar
# os.chdir('dycove-model')
# !pip install .
# os.chdir('..')

# Step 2: Install DYCOVE dependencies

In order to run a DYCOVE-ANUGA model, we need to install a few other libraries. The *pyproject.toml* file in the root folder *dycove-model/* describes the dependencies for DYCOVE.

Because DYCOVE is used for coupling with different underlying numerical models, we have separated the required Python libraries so that users of Delft3D FM, for example, don't need to install ANUGA-related dependencies.

Technically speaking, `numpy` is the only library required no matter what underlying model we are using. Of course, since we want to run ANUGA, we will need to install it. Furthermore, if we want to use DYCOVE's built-in plotting class `ModelPlotter`, we will also need to install those required libraries. In a typical Python environment, we would have to install each library included under `plot` (in *pyproject.toml*) using something like `conda install`, but Google colab actually comes with all but `netCDF4` preinstalled.

### IMPORTANT NOTE:
Outside of colab, `anuga` is most easily installed via `conda` in a dedicated conda environment; the latest directions for doing so can be found here:
https://anuga.readthedocs.io/en/latest/installation/install_anuga.html

For this notebook, we will install `anuga` using a script provided by the developers for a previous clinic, which already includes installing some dependencies.

In [ ]:
!git clone https://github.com/anuga-community/anuga-clinic.git
!/bin/bash anuga-clinic/anuga_tools/install_anuga_colab.sh

# Step 3: Import libraries

Now that these libraries have been installed within the runtime, let's import them:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import anuga

from dycove import VegetationSpecies, ANUGA_hydro

# Step 4: Create ANUGA domain

There are many ways to create an ANUGA domain, but for this example, we will use a custom Python class called `RectangSlopeDomainGenerator` that creates a rectangular model domain with a regularly-spaced, structured triangular mesh. The model bathymetry is defined via a topography function that creates a linear beach slope and dune, with a channel that bisects the dune and connects to a lagoon on the other side. The hydrodynamic forcing consists of a simple harmonic tidal signal.

`RectangSlopeDomainGenerator` is defined in a python script located in *examples/ANUGA/tide_channel/gen_anuga_domain.py*. That is where we will import it from. For a full description of all the possible inputs that users could experiment with (those arguments in the `__init__()` call), please see the class docstring in that file.

While this domain constructor class is on the heftier side, the bulk of it is related to defining custom inputs for the geometry/bathymetry of the problem. The most importants steps in creating an ANUGA domain are the following:

```python
domain = anuga.rectangular_cross_domain(...)  # there are other ways to create domains...
domain.set_quantity("elevation", topography)  # can also set elevation from a file (e.g., .ASC)
domain.set_quantity("friction", friction)     # same as above
Br = anuga.Reflective_boundary(domain)        # define the reflective BC
Bt = anuga.Time_boundary(domain, function=lambda t: [tidal_func(t), 0.0, 0.0])  # define tidal BC
domain.set_boundary({'left': Bt, 'right': Br, 'top': Br, 'bottom': Br})  # assign BCs to each domain side
```

In [ ]:
# navigate to example folder
%cd examples/ANUGA/tide_channel

from gen_anuga_domain import RectangSlopeDomainGenerator as RectangDomain

HydroDomain = RectangDomain("rectang_beach")
#HydroDomain = RectangDomain("rectang_beach", mesh_spacing=100)

Like any Python object, an `anuga` domain has various attributes and methods, some of which are assigned upon instantiation and some that are assigned later on. Attributes are static, but methods are like functions for the class/object that return or do something when you call them.

Let's look at a few key attributes and methods of an `anuga.Domain` instance:

In [ ]:
domain = HydroDomain.domain
print("ANUGA domain instance:\n", domain)
print("\nDomain name:\n", domain.get_name())  # method
print("\nDomain boundary names:\n", domain.get_boundary_tags())  # method
print("\nANUGA conserved quantity names:\n", domain.conserved_quantities)  # attribute
print("\nMesh centroid coords. as attribute:\n", domain.centroid_coordinates)  # attribute
print("\nMesh centroid coords. via method:\n", domain.get_centroid_coordinates())  # method

# BONUS: type "domain." and see the drop down menu of all possible attributes and methods

# Step 5: Instantiate a vegetation species object

In the *examples/* directory is also a `.json` file, which we use to set the attributes for the vegetation species we are modeling. When instantiating a `VegetationSpecies` object, we are also instantiating a `VegetationAttributes` dataclass object, whose job is to hold the attributes from the `.json` file.

In [ ]:
%cd dycove-model/examples/ANUGA/tide_channel/
%pwd

In [ ]:
veg_1 = VegetationSpecies("veg1.json", "veg1")

We can also print the docstrings for these classes! The docstring for ``VegetationSpecies`` tells us that there are a few optional arguments that can be implemented as well:

In [ ]:
print(VegetationSpecies.__doc__)
# help(VegetationSpecies)  # to see a printout of the entire class

We also have the option to model multiple species!

In [ ]:
from dycove import MultipleVegetationSpecies
print(MultipleVegetationSpecies.__doc__)

Let's also look at that ``VegetationAttributes`` class to see what each variable in the `.json` file means:

In [ ]:
from dycove.sim.vegetation_data import VegetationAttributes  # Need full path because VegetationAttributes is not part of the main public API

print(VegetationAttributes.__doc__)

# Step 6: Instantiate the hydrodynamic model

We pass the `domain` and `VegetationSpecies` objects to the `ANUGA` simulation class to get the model set up. The `ANUGA_hydro.py` file/module holds all logic and methods related to ANUGA, and so it is located in the *engines/* directory, where we keep all model-specific methods. *engines/* also contains *DFM_hydro.py* for all Delft3D methods, and *ANUGA_baptist.py*, which is a separate helper class that converts the vegetation state into hydrodynamic roughness (Delft3D implements its version of this internally).

The `ANUGA` simulation class inherits `HydroSimulationBase`, which actually runs the model and facilitates the coupling. Besides inheriting the base class, `ANUGA`'s only other task upon instantiation is to initialize `AnugaEngine`. `AnugaEngine` and its counterpart `DFMEngine`, handle all model-specific tasks, which are delegated from `HydroSimulationBase`.

Rather than explaining any more here, first let's instantiate the model engine:

In [ ]:
HydroModel = ANUGA_hydro.ANUGA(domain, vegetation=veg_1)

And then inspect the docstrings for these classes:

In [ ]:
print(ANUGA_hydro.ANUGA.__doc__)

print('# ------------------------------------------------------- #\n\n')

import inspect
print(inspect.getsource(ANUGA_hydro.ANUGA.__init__))

In [ ]:
print(ANUGA_hydro.AnugaEngine.__doc__)

In [ ]:
from dycove.sim.base import HydroSimulationBase
print(HydroSimulationBase.__doc__)

# Step 7: Run the simulation!

Finally, we run this simulation using a method with the same name from `HydroSimulationBase`. It is in this `run_simulation` method that we actually need to ask ourselves some of the more complex questions about ecological time stepping and scaling. But first, let's just run with the default parameters and we can think about those things while the model runs.

When the model begins time stepping below, note the first couple printed lines indicating the internally calculated `ecofac` parameter, based on the default values of `n_ets` and `veg_interval`, which are shown below in the source code for `run_simulation`.

In [ ]:
print(inspect.getsource(HydroSimulationBase.run_simulation))

In [ ]:
HydroModel.run_simulation(sim_time=3, sim_time_unit="eco-morphodynamic years")

# Step 8: Plot the results

DYCOVE contains a built-in plotter called `ModelPlotter` for visualizing 2-D model outputs. Like the DYCOVE simulation code, the plotting code also has backend classes for handling the different file and variable formats of ANUGA and Delft3D FM.

Let's make some animations! Figures will end up in the *examples/ANUGA/tide_channel/*  folder, based on our `cd` from earlier.

In [ ]:
from pathlib import Path
from dycove import plotting

plotter = plotting.ModelPlotter(
    simdir = Path('.'),
    #quantity = 'Velocity',
    quantity = 'Stem Height',
    #quantity = 'Mortality -- Total',
    #quantity = 'Fractions',
    plot_times = {  # times specified here are hydrodynamic time, not eco-morpho time
        # sim hr to start plotting
        'plotHR_0': 0*24.,
        # sim hr to stop plotting, not to exceed total sim length.
        'plotHR_f': 1*24.,  # 21 hydro days ~ 3 eco-morpho years when ecofac ~ 50
        # sim hrs between map outputs, default for ANUGA, value for DFM given in MDU file
        'mapHR_int': 1,
        # hrs between consecutive plots, cannot be less than map_output, unused if plotting vegetation
        'plotHR_int': 1,
        },
    cmap_lims = {
        'Bathymetry': (-0.5, 0.5),
        },
    animate=True,
)

plotter.run()

print("Finished creating plots!")

Of course, let's not forget to look at the docstrings for ``ModelPlotter``!
There are many custom options we can use for our plots, as well as more quantities to plot.

In [ ]:
print(plotting.ModelPlotter.__doc__)

Of course, if you would rather handle plotting on your own or make plot types that are not supported here, that is an option!

The following code is taken from the DYCOVE documentation (see top of notebook for link) under "User Guide" &rarr; "Accessing and Plotting Outputs Without `ModelPlotter`". Note that this applies to DYCOVE-ANUGA outputs, but syntax is also provided for Delft3D FM on that page.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import json
import xarray as xr
import numpy as np
from dycove.utils.array_math import cell_averaging
from dycove.utils.plotting import create_nn_interpFunc
from dycove.utils.model_loader import get_anuga_centroid_coords

# Declare model directory paths
model_dir = Path(".")
eco_dir = model_dir / "veg_output"

# Load ANUGA (or DFM) output, read X and Y coordinate arrays
map_vars = xr.load_dataset(model_dir / "rectang_beach.sww")

# Convert ANUGA vertex coordinates to centroids (may take a little time)
# DFM USERS: x_c and y_c can be pulled from the output file directly (see above)
x_c, y_c = get_anuga_centroid_coords(map_vars)

# Create interpolation function for 10-m grid. Can also provide as an argument a polygon .csv file to mask outside domain.
interp_func = create_nn_interpFunc(x_c, y_c, 10)

# Interpolate model bathymetry to use as a base map (using centroids)
z_grid = interp_func(map_vars["elevation_c"])

# Read output cohort index file, categorizing each cohort output file by ecological year and ETS
with open(eco_dir / "_cohort_files_ets_index.json", "r") as f:
    cohort_index = json.load(f)

# Loop through all saved cohort files by year and ETS, load and plot data
eco_years = sorted(cohort_index, key=int)  # output eco years in order
for year in eco_years:
    ets_list = sorted(cohort_index[year], key=int)  # ETS in order for current eco year
    for ets in ets_list:
        fractions, stem_heights = [], []
        for file in cohort_index[year][ets]:
            c = xr.load_dataset(eco_dir / (file + ".nc"))

            # Append to list all data from this ETS
            fractions.append(c["fraction"])  # each c["fraction"] is an array
            stem_heights.append(c.attrs["height"])  # each c.attrs["height"] is a scalar

        # Do weighted average based on vegetation fractions in each cell
        stemht_avg = cell_averaging(fractions, stem_heights)

        # Interpolate to grid using same interp_func as for the model elevation values
        stemht_grid = interp_func(stemht_avg)

        # Mask out non-vegetated cells
        stemht_grid = np.ma.masked_where(stemht_grid < 0.05, stemht_grid)

        # Do plotting (colorbars, etc can be added as well)
        fig, ax = plt.subplots()
        im_base = ax.imshow(z_grid, cmap="Greys_r")
        im_veg = ax.imshow(stemht_grid, cmap="Greens", vmin=0, vmax=1)
        ax.set_title(f"Stem height [m] -- eco year {year}, ETS {ets}")
        plt.show()
        plt.close()